### Feature Engineering of Attacks DataFrame

Creating the windows for the model based on the source_file column so that we divide the windows based on the file and not at random since we will be using a stride = 2 and in order to prevent overlap and data leakage on the training and testing sets. 

In [ ]:
WINDOW_SIZE = 32 
STRIDE = 2 

Take the 11 features that each CAN message contains. 

Convert the hexadecimal values inside the payload column into numerical values for the ingestion of the model. 

In [ ]:
feature_columns = ["id_int", "byte0", "byte1", "byte2", "byte3", "byte4", "byte5", "byte6", "byte7", "dlc", "dt"]

byte_columns = [f"byte{i}" for i in range(8)]

# Convert the payload column into a dataframe with the 8 bytes as columns 
attacks_df[byte_columns] = pd.DataFrame(attacks_df["payload"].tolist(), index=attacks_df.index)

# Convert the bytes from hexadecimal to decimal 
for col in byte_columns: 
    attacks_df[col] = attacks_df[col].apply(int,base=16)

Generating the sliding windows

In [ ]:
# For each source log file, sort it by timestamp, extract the features and labels, then generate the sliding_windows 

windows = []
labels = []
sources = []

for source_name, source_log in attacks_df.groupby("source_file", sort=False): 
    source_log = source_log.sort_values("timestamp")
    
    feature_matrix = source_log[feature_columns].to_numpy(dtype=np.float32)
    source_labels = source_log["label"].to_numpy()
    
    for start in range(0, len(feature_matrix) - WINDOW_SIZE + 1, STRIDE):
        
        window_features = feature_matrix[start:start + WINDOW_SIZE]
        window_labels = source_labels[start:start + WINDOW_SIZE]
        
        if (window_labels != "Benign").any(): 
            label = window_labels[window_labels != "Benign"][0]
        else: 
            label = "Benign"
            
        windows.append(window_features)
        labels.append(label)
        sources.append(source_name)
        
print(len(windows))
print(len(labels))
print(len(sources))